In [ ]:
import json
import random
import pandas as pd
import re
import os

def extract_coco_captions_by_category(json_path, num_samples_per_cat=50, existing_csv_path=None):
    print("Reading COCO JSON...")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_captions = [ann['caption'].strip() for ann in data['annotations']]
    unique_captions = list(set(all_captions))
    print(f"Total unique captions: {len(unique_captions)}")
    
    existing_prompts = set()
    if existing_csv_path and os.path.exists(existing_csv_path):
        print(f"Reading existing data '{existing_csv_path}' to create blacklist...")
        existing_df = pd.read_csv(existing_csv_path)
        # Convert to Set to optimize search speed
        if 'standard_prompt' in existing_df.columns:
            existing_prompts = set(existing_df['standard_prompt'].tolist())
            print(f"Number of already used captions: {len(existing_prompts)}")
    
    KEYWORDS = {
        "animals": ["dog", "cat", "bird", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "animal", "pet"],
        "food": ["pizza", "apple", "banana", "orange", "broccoli", "carrot", "hot dog", "donut", "cake", "food", "plate", "meal", "bread"],
        "human beings": ["man", "woman", "boy", "girl", "person", "people", "child", "kids", "guy", "lady"],
        "landscapes": ["mountain", "beach", "forest", "tree", "river", "lake", "ocean", "landscape", "sunset", "nature", "field", "sky"],
        "transport vehicles": ["car", "bus", "train", "truck", "boat", "airplane", "bike", "motorcycle", "vehicle", "street"],
        "home scenes": ["kitchen", "living room", "bedroom", "couch", "bed", "chair", "table", "tv", "indoor", "room", "desk"]
    }
    
    categorized_prompts = {cat: [] for cat in KEYWORDS.keys()}
    
    print("Categorizing captions based on keywords...")
    for caption in unique_captions:
        # Skip regex matching for blacklisted captions to save computation
        if caption in existing_prompts:
            continue
            
        caption_lower = caption.lower()
        for cat, keywords in KEYWORDS.items():
            if any(re.search(r'\b' + re.escape(kw) + r'\b', caption_lower) for kw in keywords):
                categorized_prompts[cat].append(caption)
                break 
                
    records = []
    
    # Fixed seed maintained for consistent sampling initially
    random.seed(42) 
    
    for cat, prompts in categorized_prompts.items():
        actual_samples = min(num_samples_per_cat, len(prompts))
        sampled = random.sample(prompts, actual_samples)
        
        print(f"[{cat}] Extracted: {actual_samples} (Total candidates: {len(prompts)})")
        
        for p in sampled:
            records.append({
                'category': f'Benign_{cat.replace(" ", "_")}', 
                'standard_prompt': p
            })
            
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} non-overlapping new prompts.")
    
    return df


In [ ]:
df_new_batch = extract_coco_captions_by_category(
    json_path='captions_val2017.json', 
    num_samples_per_cat=400, 
    existing_csv_path='benign_coco_categorized.csv'
)

df_new_batch.to_csv("benign_coco_categorized_new_batch.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_new_batch